# Detecção de Fraude 

In [80]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
 )
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1) Carregar dados e definir Feature Engineering (sem leakage)

In [81]:
DATA_PATH = Path("./creditcard.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../creditcard.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("creditcard.csv não encontrado")

df = pd.read_csv(DATA_PATH)

v_columns = [f"V{i}" for i in range(1, 29)]
required_columns = ["Time", "Amount", *v_columns, "Class"]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {missing_columns}")

original_feature_columns = ["Time", "Amount", *v_columns]

train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df["Class"], random_state=RANDOM_STATE
)
train_fit_df, val_df = train_test_split(
    train_df, test_size=0.25, stratify=train_df["Class"], random_state=RANDOM_STATE
)

y_train = train_fit_df["Class"].astype(int)
y_val = val_df["Class"].astype(int)
y_test = test_df["Class"].astype(int)

print(f"Dataset: {df.shape}")
print(f"Treino (fit): {train_fit_df.shape}")
print(f"Validação:    {val_df.shape}")
print(f"Teste:        {test_df.shape}")

Dataset: (284807, 31)
Treino (fit): (170883, 31)
Validação:    (56962, 31)
Teste:        (56962, 31)


## 1.1) Feature Engineering

In [82]:
def feature_engineering(input_df: pd.DataFrame) -> pd.DataFrame:
    engineered_df = input_df.copy()

    engineered_df["Amount_log"] = np.log1p(engineered_df["Amount"])

    seconds_in_day = 24 * 60 * 60
    engineered_df["Hour"] = ((engineered_df["Time"] % seconds_in_day) // 3600).astype(int)
    angle = 2 * np.pi * (engineered_df["Time"] % seconds_in_day) / seconds_in_day
    engineered_df["Time_sin"] = np.sin(angle)
    engineered_df["Time_cos"] = np.cos(angle)
    engineered_df["Day"] = (engineered_df["Time"] // seconds_in_day).astype(int)

    v_data = engineered_df[v_columns]
    engineered_df["V_mean"] = v_data.mean(axis=1)
    engineered_df["V_std"] = v_data.std(axis=1)
    engineered_df["V_abs_max"] = v_data.abs().max(axis=1)
    engineered_df["V_l2_norm"] = np.sqrt((v_data**2).sum(axis=1))

    return engineered_df

In [83]:
preview_fe_df = feature_engineering(df.head(5))

new_features = [
    "Amount_log",
    "Hour",
    "Time_sin",
    "Time_cos",
    "Day",
    "V_mean",
    "V_std",
    "V_abs_max",
    "V_l2_norm",
]

print(f"Total de colunas antes do FE: {df.shape[1]}")
print(f"Total de colunas depois do FE: {preview_fe_df.shape[1]}")
print("\nNovas features criadas:")
print(new_features)

print("\nExemplo (5 primeiras linhas) das novas features:")
display(preview_fe_df[new_features])

Total de colunas antes do FE: 31
Total de colunas depois do FE: 40

Novas features criadas:
['Amount_log', 'Hour', 'Time_sin', 'Time_cos', 'Day', 'V_mean', 'V_std', 'V_abs_max', 'V_l2_norm']

Exemplo (5 primeiras linhas) das novas features:


,Amount_log,Hour,Time_sin,Time_cos,Day,V_mean,V_std,V_abs_max,V_l2_norm
0,5.014760,0,0.000000,1.000000,0,0.110063,0.744389,2.536347,3.911559
1,1.305626,0,0.000000,1.000000,0,0.158562,0.488729,1.612727,2.674524
2,5.939276,0,0.000073,1.000000,0,0.038975,1.169522,2.890083,6.080512
3,4.824306,0,0.000073,1.000000,0,-0.086057,0.819854,1.965775,4.284356
4,4.262539,0,0.000145,1.000000,0,0.192097,0.657631,1.548718,3.565131


## 1.2) Visualizar como ficou após o Feature Engineering

## 2) Preparar baseline e conjunto com novas features

In [84]:
train_fe_df = feature_engineering(train_fit_df)
val_fe_df = feature_engineering(val_df)
test_fe_df = feature_engineering(test_df)

X_train_baseline = train_fit_df.drop(columns=["Class"]).copy()
X_val_baseline = val_df.drop(columns=["Class"]).copy()
X_test_baseline = test_df.drop(columns=["Class"]).copy()

X_train_fe = train_fe_df.drop(columns=["Class"]).copy()
X_val_fe = val_fe_df.drop(columns=["Class"]).copy()
X_test_fe = test_fe_df.drop(columns=["Class"]).copy()

n_pos_train = int((y_train == 1).sum())
n_neg_train = int((y_train == 0).sum())
scale_pos_weight = n_neg_train / n_pos_train

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "random_state": RANDOM_STATE,
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "n_jobs": -1,
}

threshold_grid = np.linspace(0.05, 0.95, 19)

print(f"No treino (fit): negativos={n_neg_train}, positivos={n_pos_train}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")
print(f"Features baseline: {X_train_baseline.shape[1]}")
print(f"Features com engenharia: {X_train_fe.shape[1]}")

No treino (fit): negativos=170588, positivos=295
scale_pos_weight: 578.26
Features baseline: 30
Features com engenharia: 39


## 3) Treinar experimentos comparáveis e analisar thresholds

In [85]:
# Treinamento XGBoost
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)

# Threshold por F1 na validação
y_val_proba = xgb_model.predict_proba(X_val)[:, 1]
candidate_thresholds = np.linspace(0.01, 0.99, 99)
f1_scores = [f1_score(y_val, (y_val_proba >= t).astype(int), zero_division=0) for t in candidate_thresholds]
chosen_threshold = float(candidate_thresholds[int(np.argmax(f1_scores))])
val_pr_auc = average_precision_score(y_val, y_val_proba)
val_roc_auc = roc_auc_score(y_val, y_val_proba)

print(f"Validação PR-AUC: {val_pr_auc:.4f}")
print(f"Validação ROC-AUC: {val_roc_auc:.4f}")
print(f"Threshold escolhido (F1): {chosen_threshold:.2f}")

Validação PR-AUC: 0.8225
Validação ROC-AUC: 0.9738
Threshold escolhido (F1): 0.82


## 4) Salvar modelo para API

In [86]:
artifacts_dir = Path("./artifacts")
if not artifacts_dir.exists():
    artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = artifacts_dir / "xgboost_model.joblib"
metadata_path = artifacts_dir / "xgboost_metadata.json"

joblib.dump(xgb_model, model_path)

metadata = {
    "model_version": "xgboost-v1",
    "threshold": float(chosen_threshold),
    "features": feature_columns,
    "validation_metrics": {
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "pr_auc": float(test_pr_auc),
        "roc_auc": float(test_roc_auc),
    },
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Modelo salvo em: {model_path}")
print(f"Metadados salvos em: {metadata_path}")

Modelo salvo em: ../artifacts/xgboost_model.joblib
Metadados salvos em: ../artifacts/xgboost_metadata.json


## 5) Avaliar no teste

In [87]:
y_test_proba = xgb_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= chosen_threshold).astype(int)

test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_pr_auc = average_precision_score(y_test, y_test_proba)
test_roc_auc = roc_auc_score(y_test, y_test_proba)

print("Métricas no teste:")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1:        {test_f1:.4f}")
print(f"PR-AUC:    {test_pr_auc:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")

Métricas no teste:
Precision: 0.9000
Recall:    0.8265
F1:        0.8617
PR-AUC:    0.8664
ROC-AUC:   0.9797


In [88]:
def analyze_transaction(transaction: dict | pd.Series | pd.DataFrame) -> dict:
    if isinstance(transaction, dict):
        tx_df = pd.DataFrame([transaction])
    elif isinstance(transaction, pd.Series):
        tx_df = transaction.to_frame().T
    elif isinstance(transaction, pd.DataFrame) and len(transaction) == 1:
        tx_df = transaction.copy()
    else:
        raise ValueError("Use dict, Series ou DataFrame com 1 linha")

    tx_df = tx_df.reindex(columns=feature_columns)
    if tx_df.isna().any().any():
        raise ValueError("Transação com colunas faltando")

    fraud_probability = float(xgb_model.predict_proba(tx_df)[0, 1])
    decision = "fraud" if fraud_probability >= chosen_threshold else "legitimate"

    return {
        "decision": decision,
        "fraud_probability": fraud_probability,
        "threshold": float(chosen_threshold),
    }

# Exemplo de validação final com transação fraudulenta real do teste (Class=1)
fraud_indices = y_test[y_test == 1].index

if len(fraud_indices) == 0:
    print("Não há transação fraudulenta no conjunto de teste.")
else:
    fraud_idx = fraud_indices[0]
    fraudulent_transaction = X_test.loc[fraud_idx]
    true_label = int(y_test.loc[fraud_idx])

    print(fraudulent_transaction)

    example_result = analyze_transaction(fraudulent_transaction)
    print("Exemplo de transação fraudulenta (teste):")
    print(f"Rótulo real: {true_label}")
    print(json.dumps(example_result, indent=2, ensure_ascii=False))

Time     57,007.000000
V1           -1.271244
V2            2.462675
V3           -2.851395
V4            2.324480
V5           -1.372245
V6           -0.948196
V7           -3.065234
V8            1.166927
V9           -2.268771
V10          -4.881143
V11           2.255147
V12          -4.686387
V13           0.652375
V14          -6.174288
V15           0.594380
V16          -4.849692
V17          -6.536521
V18          -3.119094
V19           1.715494
V20           0.560478
V21           0.652941
V22           0.081931
V23          -0.221348
V24          -0.523582
V25           0.224228
V26           0.756335
V27           0.632800
V28           0.250187
Amount        0.010000
Name: 77348, dtype: float64
Exemplo de transação fraudulenta (teste):
Rótulo real: 1
{
  "decision": "fraud",
  "fraud_probability": 0.9999340772628784,
  "threshold": 0.8200000000000001
}
